# MODULE04 미니프로젝트04 · 2조 · Colab에서 Streamlit 앱 열기

정유·석유화학 여신 계기판(Streamlit)을 Colab에서 띄우고 **앱 주소(https://….trycloudflare.com)** 를 출력합니다. 포함된 CSV로 실행하므로 **API 키 없이** 동작합니다.

**준비 (둘 중 하나)**
1. **드라이브 방식(권장)**: 압축을 푼 `MODULE04_미니프로젝트04_2조` 폴더를 **`내 드라이브` 맨 위에 이름 그대로** 올립니다. (다른 이름이면 ① `PROJECT_NAME`만 수정)
2. **zip 업로드 방식**: 드라이브에 폴더가 없으면 ② 셀이 zip 파일 선택 버튼을 띄웁니다.

**선택 기능 키 (없어도 됨)**: 왼쪽 열쇠 아이콘(보안 비밀)에 `DART_API_KEY`(선택 기업 데이터 업데이트 버튼), `OPENAI_API_KEY`(챗봇)를 넣고 `노트북 액세스`를 켜면 환경변수로만 전달합니다.

**실행**: `런타임 → 모두 실행` → ② 셀 아래 `▶ 앱 주소` 클릭

## ① 설정

In [ ]:
PROJECT_NAME = "MODULE04_미니프로젝트04_2조"   # 내 드라이브에 올린 프로젝트 폴더 이름
APP_FILE = "app/app.py"
PORT = 8501

## ② Streamlit 앱 실행

- 앱·데이터를 드라이브 밖 `/content/app_run`으로 복사해 실행 (드라이브에서 바로 띄우면 파일 감시 때문에 멈출 수 있음)
- 외부 주소는 cloudflared 무료 터널(계정 불필요). 열리기까지 10~20초 걸릴 수 있음

In [ ]:
import os, sys, glob, shutil, subprocess, time, re, zipfile, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUN = Path("/content/app_run")

# 1) 프로젝트 폴더 찾기: 드라이브 -> 없으면 zip 업로드
BASE = None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    cand = Path("/content/drive/MyDrive") / PROJECT_NAME
    if (cand / APP_FILE).exists():
        BASE = cand
if BASE is None and IN_COLAB:
    from google.colab import files
    print(f"내 드라이브에 '{PROJECT_NAME}' 폴더가 없습니다. 프로젝트 zip 파일을 선택해 주세요.")
    up = files.upload()
    tmp = Path("/content/upload")
    if tmp.exists(): shutil.rmtree(tmp)
    with zipfile.ZipFile("/content/" + next(iter(up))) as z:
        z.extractall(tmp)
    hits = glob.glob(str(tmp / "**" / APP_FILE), recursive=True)
    assert hits, f"zip 안에서 {APP_FILE} 를 찾지 못했습니다."
    BASE = Path(hits[0][: -len(APP_FILE)])
assert BASE is not None, "Colab에서 실행해 주세요. 로컬은 run_app.bat 사용"
print("프로젝트 폴더:", BASE)

# 2) 드라이브 밖으로 복사 (app, data, 테마 설정)
if RUN.exists(): shutil.rmtree(RUN)
shutil.copytree(BASE / "app", RUN / "app")
shutil.copytree(BASE / "data", RUN / "data")
for extra in (".streamlit", "source"):          # 테마 설정, '데이터 업데이트' 버튼용 수집 코드
    if (BASE / extra).exists():
        shutil.copytree(BASE / extra, RUN / extra)
for f in ["metrics.csv", "quarterly_metrics.csv", "events.csv"]:
    print(("  OK   " if (RUN / "data/processed" / f).exists() else "  없음 ") + "data/processed/" + f)

# 3) 패키지 설치
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(BASE / "requirements.txt")], capture_output=True, text=True)
if r.returncode != 0:
    print("requirements.txt 설치 실패 -> 앱에 필요한 패키지만 설치")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit", "pandas", "plotly"], check=True)

# 4) 선택 기능용 키: Colab 왼쪽 열쇠(보안 비밀)에 등록했을 때만 환경변수로 전달 (파일에 쓰지 않음)
env = dict(os.environ)
for name in ("DART_API_KEY", "OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value: env[name] = value
    except Exception:
        pass

# 5) Streamlit 실행 (백그라운드)
subprocess.run(["pkill", "-f", "streamlit run"], check=False)
st_log = open("/content/streamlit.log", "w")
app_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", APP_FILE, "--server.port", str(PORT),
                             "--server.headless", "true", "--server.fileWatcherType", "none",
                             "--browser.gatherUsageStats", "false"],
                            cwd=RUN, env=env, stdout=st_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/streamlit.log").read()[-2000:]); raise RuntimeError("Streamlit이 시작되지 않았습니다 (위 로그 확인)")

# 6) 외부 주소 만들기 (cloudflared 무료 터널)
CF = Path("/content/cloudflared")
if not CF.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    CF.chmod(0o755)
subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
cf_log = open("/content/cloudflared.log", "w")
tunnel = subprocess.Popen([str(CF), "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                          stdout=cf_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    found = [u for u in re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read()) if "//api." not in u]
    if found: url = found[0]; break
    if tunnel.poll() is not None: break
    time.sleep(1)
print("▶ 앱 주소:", url or "(터널 주소를 못 받음, 아래 대체 주소 사용)")
try:                                   # 대체: Colab 자체 프록시 주소 (같은 브라우저에서만 열림)
    from google.colab.output import eval_js
    print("  대체 주소:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
except Exception:
    pass
print("※ 주소가 열리기까지 10~20초 걸릴 수 있음 · Colab을 닫으면 주소도 끊김 · 끝낼 때는 ③ 셀 실행")

## ③ 종료

In [ ]:
try:
    app_proc.terminate(); tunnel.terminate()
except NameError:
    pass
print("앱과 터널을 종료했습니다.")

---
**안 될 때**
- `zip 안에서 … 찾지 못했습니다`: zip 안에 `app/app.py`가 있는지 확인
- 주소를 열었는데 하얀 화면만: 10~20초 뒤 새로고침, 그래도 안 되면 `대체 주소` 사용
- `Streamlit이 시작되지 않았습니다`: 출력된 로그의 마지막 줄 확인 (대부분 CSV 누락이나 경로 문제)
- 참고용이며 실제 은행 심사 기준이 아님 · 투자 권유 아님